# **SOM Preprocessing and Training*** #

## **Open the saved zarr file** ##

In [ ]:
ds = xr.open_zarr('/home/scratch/483_Project_Data/cape_cin_NARR.zarr/')
ds

## **Standardize the data for SOM training** ##

In [ ]:
CAPE = ds['cape'].values #Calling the CAPE and CIN values in the dataset
CIN = ds['cin'].values

N, ny, nx = CAPE.shape
P = ny * nx

X = np.concatenate([CAPE.reshape(N, P), CIN.reshape(N, P)], axis=1)

print(X.shape) #Printing X to ensure the shape is correct...all values should be above 0

In [ ]:
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

In [ ]:
X_cape = ds["cape"].values.reshape(ds.sizes["time"], -1)
X_cin  = ds["cin"].values.reshape(ds.sizes["time"], -1)

X = np.concatenate([X_cape, X_cin], axis=1).astype(np.float32)

#Test to ensure data is working correctly
print("Raw X shape:", X.shape)
print("Raw NaNs:", np.isnan(X).sum())
print("Raw finite:", np.isfinite(X).all())

In [ ]:
X[~np.isfinite(X)] = np.nan
X[np.abs(X) > 1e20] = np.nan

# keep columns that are not entirely NaN
good_cols = ~np.isnan(X).all(axis=0)
X = X[:, good_cols]

#Test to ensure data is working correctly
print("After all-NaN column drop:", X.shape)

In [ ]:
'''
The dataset we use contains several nan values, to rectify this we are removing all of the nans and restandardizing the data.

If the nans are not removed, all values turn to nan which will not work when fed into the SOM

'''

col_mean = np.nanmean(X, axis=0)
col_mean = np.nan_to_num(col_mean, nan=0.0)

inds = np.where(np.isnan(X))
X[inds] = col_mean[inds[1]]

std = np.std(X, axis=0)
X = X[:, std > 0]

#Test to ensure data is working correctly
print("Clean X shape:", X.shape)
print("NaNs:", np.isnan(X).sum())
print("Finite:", np.isfinite(X).all())

In [ ]:
scaler = MinMaxScaler()
Xz = scaler.fit_transform(X)

#Test to ensure data is working correctly
print("Xz shape:", Xz.shape)
print("NaNs in Xz:", np.isnan(Xz).sum())
print("Finite Xz:", np.isfinite(Xz).all())

## **Configure SOM** ##

In [ ]:
'''
Settings were chose arbitrarily but there are methods from which to select "statistically accurate" settings

Once the SOM has completed training, the MiniSom package provides a quantization error. 

Note that the higher the settings you set, the longer the model wi

'''

som_x = 5 #number of columns
som_y = 5 #number of rows
num_iterations = 4000 #amount of time the model reasseses

som = MiniSom( #The settings are fed into the MiniSom package and applied automatically
    x=som_x,
    y=som_y,
    input_len=Xz.shape[1],
    sigma=2.0,
    learning_rate=1,
    neighborhood_function='gaussian',
    random_seed=0
)

som.random_weights_init(Xz)
som.train_random(Xz, num_iterations, verbose=True)

bmus = np.array([som.winner(x) for x in Xz])
print(bmus.shape)